In [61]:
import time 
import warnings
import numpy as np 
import pandas as pd
import requests
import matplotlib.pyplot as plt

import torch
import torch.nn as nn # for the neuro network 
from torch.utils.data import TensorDataset, DataLoader 
from sklearn.linear_model import Ridge 

In [62]:
import requests, pandas as pd, time

url = "https://api.binance.com/api/v3/klines"
res = requests.get(url, params={
    "symbol": "BTCUSDT", "interval": "1h", "limit": 1000
}).json()

# Print the first row — see what Binance sends you
print(res[0])

[1786946400000, '63520.18000000', '63717.17000000', '63477.95000000', '63667.99000000', '312.69187000', 1786949999999, '19888108.13010130', 60567, '146.54017000', '9320329.66449190', '0']


In [63]:
columns = [
    "open_time", "open", "high", "low", "close", "volume",
    "close_time", "quote_asset_volume", "number_of_trades",
    "taker_buy_base_asset_volume", "taker_buy_quote_asset_volume", "ignore"
]
df = pd.DataFrame(res,columns= columns)
print(df.head())

       open_time            open            high             low  \
0  1786946400000  63520.18000000  63717.17000000  63477.95000000   
1  1786950000000  63668.00000000  63668.00000000  63500.00000000   
2  1786953600000  63514.45000000  63545.08000000  63350.00000000   
3  1786957200000  63380.83000000  63428.00000000  63295.00000000   
4  1786960800000  63323.97000000  63694.26000000  63300.25000000   

            close        volume     close_time quote_asset_volume  \
0  63667.99000000  312.69187000  1786949999999  19888108.13010130   
1  63514.45000000  696.41924000  1786953599999  44260624.87924190   
2  63380.84000000  520.36494000  1786957199999  33011742.74853170   
3  63323.96000000  311.26926000  1786960799999  19719661.42280220   
4  63692.21000000  683.00101000  1786964399999  43379286.46763790   

   number_of_trades taker_buy_base_asset_volume taker_buy_quote_asset_volume  \
0             60567                146.54017000             9320329.66449190   
1             54

In [64]:
df = df.drop(columns = ["ignore"])

In [65]:
print(df.head())

       open_time            open            high             low  \
0  1786946400000  63520.18000000  63717.17000000  63477.95000000   
1  1786950000000  63668.00000000  63668.00000000  63500.00000000   
2  1786953600000  63514.45000000  63545.08000000  63350.00000000   
3  1786957200000  63380.83000000  63428.00000000  63295.00000000   
4  1786960800000  63323.97000000  63694.26000000  63300.25000000   

            close        volume     close_time quote_asset_volume  \
0  63667.99000000  312.69187000  1786949999999  19888108.13010130   
1  63514.45000000  696.41924000  1786953599999  44260624.87924190   
2  63380.84000000  520.36494000  1786957199999  33011742.74853170   
3  63323.96000000  311.26926000  1786960799999  19719661.42280220   
4  63692.21000000  683.00101000  1786964399999  43379286.46763790   

   number_of_trades taker_buy_base_asset_volume taker_buy_quote_asset_volume  
0             60567                146.54017000             9320329.66449190  
1             5419

In [66]:
df["timetamp"] = pd.to_datetime(df["open_time"] , unit="ms" , utc=True )

for c in [
    "open_time", "open", "high", "low", "close", "volume",
    "close_time", "quote_asset_volume", "number_of_trades",
    "taker_buy_base_asset_volume", "taker_buy_quote_asset_volume"
]: 
    df[c] = df[c].astype(float) # so here we can convert the data type into float


In [67]:
df = df.sort_values("timetamp").drop_duplicates("timetamp").reset_index(drop=True) 
# so here the sort values make the time sorted in stead of make the 11 pm and 1 am look close it make it look distant as it should so the module can understand 
# and for the drop it drop duplicates 
#utc = true for when i drop the column 5 it get replaced with the next one instead of being null 

In [68]:
print(df.columns.tolist())

['open_time', 'open', 'high', 'low', 'close', 'volume', 'close_time', 'quote_asset_volume', 'number_of_trades', 'taker_buy_base_asset_volume', 'taker_buy_quote_asset_volume', 'timetamp']


In [69]:
print(df[["volume", "quote_asset_volume"]].describe())
print(f"Mean price implied: {(df['quote_asset_volume'] / df['volume']).mean():.0f}")

print(df["number_of_trades"].describe())

violations = (df["taker_buy_base_asset_volume"] > df["volume"] * 1.001).sum()
print(f"Violations: {violations}")   

df["taker_buy_ratio"] = (
    df["taker_buy_base_asset_volume"] / (df["volume"] + 1e-8)
)
print(df["taker_buy_ratio"].describe())  

            volume  quote_asset_volume
count  1000.000000        1.000000e+03
mean    718.035925        5.617932e+07
std     691.211095        5.314957e+07
min      14.955080        1.264869e+06
25%     339.663900        2.642291e+07
50%     523.746245        4.099798e+07
75%     837.448715        6.485113e+07
max    9262.869650        6.284078e+08
Mean price implied: 78358
count      1000.000000
mean     139658.683000
std      114274.898118
min        2904.000000
25%       64843.750000
50%      105724.000000
75%      172776.500000
max      912256.000000
Name: number_of_trades, dtype: float64
Violations: 0
count    1000.000000
mean        0.496413
std         0.081400
min         0.236964
25%         0.444249
50%         0.495212
75%         0.552870
max         0.796411
Name: taker_buy_ratio, dtype: float64


In [70]:
df["return"] = ((df["close"]-df["open"] )/df["open"] )*100

In [71]:
df["hourly_range"] = (df["high"] - df["low"]) / df["low"] * 100

In [72]:
#close-to-close return:
#(Ci - Ci-1) / Ci-1 * 100 
df["rt_cc"] = ((df["close"] - df["close"].shift(1)) / df["close"].shift(1)) * 100

In [73]:
df["range_ma_24"] = df["hourly_range"].rolling(24).mean() # moving average of 24 hour range 
df["range_ma_168"] = df["hourly_range"].rolling(168).mean() # moving average of 168 hour range

In [74]:
df["volatility_24"] = df["rt_cc"].rolling(24).std()

In [75]:
df["volume_ma_24"] = df["volume"].rolling(24).mean()
df["volume_ratio"] = df["volume"] / (df["volume_ma_24"] + 1e-8)

In [76]:
#cyclical time encoding:(its an important concept for seprating between the 23 pm  and 1 am )
tau = 2 * np.pi  # Full circle constant (2 * pi)

hour = df["timetamp"].dt.hour
dow = df["timetamp"].dt.dayofweek

df[["hour_sin", "hour_cos"]] = np.column_stack(
    [np.sin(tau * hour / 24), np.cos(tau * hour / 24)]
)

df[["dow_sin", "dow_cos"]] = np.column_stack(
    [np.sin(tau * dow / 7), np.cos(tau * dow / 7)]
)

In [77]:
df["volume_change"] = df["volume"].pct_change() * 100

df["range_ma_720"] = df["hourly_range"].rolling(720).mean()
df["volatility_168"] = df["rt_cc"].rolling(168).std()

df["ma_24_price"]  = df["close"].rolling(24).mean()
df["ma_168_price"] = df["close"].rolling(168).mean()
df["distance_ma_24"]  = (df["close"] - df["ma_24_price"])  / (df["ma_24_price"]  + 1e-8)
df["distance_ma_168"] = (df["close"] - df["ma_168_price"]) / (df["ma_168_price"] + 1e-8)

for lag in (1, 2, 3, 6, 12):
    df[f"range_lag_{lag}"] = df["hourly_range"].shift(lag)

df["is_weekend"] = (df["timetamp"].dt.dayofweek >= 5).astype(float)

In [78]:
df["taker_buy_ratio"] = (
    df["taker_buy_base_asset_volume"] / (df["volume"] + 1e-8)
)
df["order_flow_imbalance"] = (
    (2 * df["taker_buy_base_asset_volume"] - df["volume"])
    / (df["volume"] + 1e-8)
)
df["avg_trade_size"]     = df["volume"] / (df["number_of_trades"] + 1e-8)
df["avg_trade_size_usd"] = df["quote_asset_volume"] / (df["number_of_trades"] + 1e-8)

In [79]:
df["vol_regime"]      = df["volatility_24"] / (df["volatility_168"] + 1e-8)
df["range_vs_ma_168"] = df["hourly_range"] / (df["range_ma_168"] + 1e-8)
df["range_vs_ma_720"] = df["hourly_range"] / (df["range_ma_720"] + 1e-8)

df["high_vol_streak"] = (
    (df["hourly_range"] > df["range_ma_24"]).rolling(6).sum()
)

df["roll_max_720"]  = df["close"].rolling(720).max()
df["drawdown_720"]  = (df["close"] - df["roll_max_720"]) / (df["roll_max_720"] + 1e-8)

df["return_skew_168"] = df["rt_cc"].rolling(168).skew()

In [80]:
df["trade_count_change"] = df["number_of_trades"].pct_change() * 100

In [81]:
df["vwap_hour"]     = df["quote_asset_volume"] / (df["volume"] + 1e-8)
df["close_vs_vwap"] = (df["close"] - df["vwap_hour"]) / (df["vwap_hour"] + 1e-8) * 100

In [82]:
df["ofi_ma_24"]        = df["order_flow_imbalance"].rolling(24).mean()
df["ofi_std_24"]       = df["order_flow_imbalance"].rolling(24).std()
df["taker_buy_ma_24"]  = df["taker_buy_ratio"].rolling(24).mean()
df["trade_size_ma_24"] = df["avg_trade_size"].rolling(24).mean()
df["trade_size_ratio"] = df["avg_trade_size"] / (df["trade_size_ma_24"] + 1e-8)

In [83]:
for lag in (1, 2, 3):
    df[f"trade_ratio_lag_{lag}"] = df["taker_buy_ratio"].shift(lag)

In [84]:
for lag in (1, 2, 3, 6, 12):
    df[f"ofi_lag_{lag}"] = df["order_flow_imbalance"].shift(lag)

In [85]:
df["next_hour_range"] = df["hourly_range"].shift(-1)

In [86]:
df["target"] = df["next_hour_range"] / (df["range_ma_24"] + 1e-8)
df["target"] = df["target"].clip(0, 100)

['open_time', 'open', 'high', 'low', 'close', 'volume', 'close_time', 'quote_asset_volume', 'number_of_trades', 'taker_buy_base_asset_volume', 'taker_buy_quote_asset_volume', 'timetamp']

In [87]:
def fetch_funding(start_ms, end_ms):
    records, cur_time = [], start_ms
    while cur_time < end_ms:
        res = requests.get("https://fapi.binance.com/fapi/v1/fundingRate",
                           params={"symbol": "BTCUSDT",
                                   "startTime": cur_time, "limit": 1000},
                           timeout=10)
        data = res.json()
        if not data:
            break
        records.extend(data)
        cur_time = data[-1]["fundingTime"] + 1
        time.sleep(0.02)

    df_f = pd.DataFrame(records)
    df_f["timestamp"] = pd.to_datetime(df_f["fundingTime"], unit="ms", utc=True)
    df_f["funding_rate"] = df_f["fundingRate"].astype(float)
    df_f["funding_rate_change"] = df_f["funding_rate"].diff()
    mean_8h = df_f["funding_rate"].rolling(21, min_periods=7).mean()
    std_8h  = df_f["funding_rate"].rolling(21, min_periods=7).std() + 1e-8
    df_f["z_funding"] = (df_f["funding_rate"] - mean_8h) / std_8h
    return df_f[["timestamp", "funding_rate", "funding_rate_change", "z_funding"]]

start_ms = int(df["timetamp"].min().timestamp() * 1000)
end_ms   = int(df["timetamp"].max().timestamp() * 1000)
df_funding = fetch_funding(start_ms, end_ms)

df = pd.merge_asof(df.sort_values("timetamp"),
                   df_funding.sort_values("timestamp"),
                   left_on="timetamp", right_on="timestamp",
                   direction="backward")

df[["funding_rate", "funding_rate_change", "z_funding"]] = (
    df[["funding_rate", "funding_rate_change", "z_funding"]].ffill().fillna(0)
)

cols_funding = ["funding_rate", "funding_rate_change", "z_funding"]

In [88]:
cols_ohlcv = [
    "return",              
    "hourly_range",       
    "rt_cc",            
    "volume_change",       
    "volume_ratio",    
    "range_ma_24",
    "range_ma_168",
    "range_ma_720",
    "volatility_24",
    "volatility_168",
    "distance_ma_24",
    "distance_ma_168",
    "hour_sin", "hour_cos",
    "dow_sin",  "dow_cos",
    "is_weekend",
    "range_lag_1", "range_lag_2", "range_lag_3",
    "range_lag_6", "range_lag_12",
    "vol_regime",
    "range_vs_ma_168",
    "range_vs_ma_720",
    "high_vol_streak",
    "drawdown_720",
    "return_skew_168",
]
cols_orderflow = [
    "taker_buy_ratio",         
    "order_flow_imbalance",   
    "close_vs_vwap",          
    "avg_trade_size",         
    "avg_trade_size_usd",     
    "trade_size_ratio",       
    "trade_count_change", 
    "ofi_ma_24",
    "ofi_std_24",
    "taker_buy_ma_24",
    "trade_size_ma_24",
    "ofi_lag_1", "ofi_lag_2", "ofi_lag_3", "ofi_lag_6", "ofi_lag_12",
    "trade_ratio_lag_1", "trade_ratio_lag_2", "trade_ratio_lag_3",
]

cols_funding = ["funding_rate", "funding_rate_change", "z_funding"]

feature_cols = cols_ohlcv + cols_orderflow + cols_funding